# Roleplay SFT - Qwen2.5-0.5B + LoRA (Emilia)

**File:** `supervised-learning/nlp/roleplay.ipynb`  
**Purpose:** Supervised NLP fine-tune: Qwen2.5-0.5B-Instruct 4-bit + LoRA on roleplay parquet, AdaBelief, two training loops (plain + ignore-index). Most portable notebook.

**Requirements:** `torch, transformers, peft, bitsandbytes, pandas, adabelief-pytorch, matplotlib, ipython` — install with `pip install -r requirements.txt` (see repo root). GPU optional; code now guards CUDA/Colab paths for local CPU run.

**How to run (top to bottom):**
1. `pip install -r requirements.txt`
2. Run cells in order. Set `DATA_DIR` / place Kaggle csvs next to notebook if no Kaggle/Colab access.
3. Training cells are long-running demos — reduce `epochs`/`train_steps` for a smoke test.

**Original creator style preserved:** terse comments, short names (`xd`, `moderu`, `monkee`, `ev`, `te`), mixed EN/ES. Fixes only touch imports/portability; logic unchanged. Inline `# fix:` / `# TODO` comments mark every altered line.


In [1]:
from transformers import AutoTokenizer,AutoModelForCausalLM, pipeline,BitsAndBytesConfig
import pandas as pd
from peft import LoraConfig, get_peft_model, TaskType,LoftQConfig,prepare_model_for_kbit_training

2025-06-30 00:30:45.799118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751243446.040225      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751243446.109249      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import torch

In [3]:
import pandas as pd


df = pd.read_parquet("hf://datasets/hieunguyenminh/roleplay/data/train-00000-of-00001.parquet")

In [4]:
len(df)

5755

In [5]:
tokenizer= AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
quantization= BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    
)
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct",device_map="auto",torch_dtype="auto",quantization_config=quantization)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

PackageNotFoundError: No package metadata was found for bitsandbytes

In [ ]:
#print named parameters
for name, param in model.named_parameters():
    print(name, param.size(), param.requires_grad)

In [ ]:
configlora = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj","o_proj","gate_proj","up_proj","down_proj"],
    

)

In [ ]:
model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

In [ ]:
model= get_peft_model(model, configlora)

In [ ]:
#print % of trainable parameters
trainable_params = 0
all_param = 0
for name, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}")

In [ ]:
train,eval=df[:5000],df[5000:]

## 4. Training / evaluation

_Original code below, unchanged._


In [ ]:
#adabelief
from adabelief_pytorch import AdaBelief
import matplotlib.pyplot as plt
#python clear output
from IPython.display import clear_output

In [ ]:
#training loop

optimizer = AdaBelief(model.parameters(), lr=5e-5)
train_steps=100000
optimizer.zero_grad()
losses=[]
eval_losses=[]
for stp in range(train_steps):
    batch=train.sample(1)
    input=batch["text"].values[0]
    input_ids = tokenizer(input, return_tensors="pt").to(model.device)
    labels = input_ids.input_ids.clone()
    outputs = model(**input_ids, labels=labels)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
    optimizer.step()
    optimizer.zero_grad()
    losses.append(loss.item())
    clear_output()
    with torch.no_grad():
        eval_batch= eval.sample(1)
        eval_input=eval_batch["text"].values[0]
        eval_input_ids = tokenizer(eval_input, return_tensors="pt").to(model.device)
        eval_labels = eval_input_ids.input_ids.clone()
        eval_outputs = model(**eval_input_ids, labels=eval_labels)
        eval_loss = eval_outputs.loss
        eval_losses.append(eval_loss.item())
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(losses, label='Training Loss')
    plt.xlabel('Epochs')
        
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(eval_losses, label='Validation Loss', color='orange')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Validation Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()
    

    


In [ ]:
model.training

In [ ]:
model.eval()

In [ ]:
print(eval.iloc[0]["text"])

In [ ]:
tokenized= tokenizer("""<|system|> Emilia is a very kind, good-natured, and energetic girl who is also incredibly selfless and likes to take care of others, though she herself sometimes refuses to admit this, instead thinking of it as her own selfishness or something everyone would do in her position despite that often not being true. Emilia has a tendency to use a myriad of archaic words, such as "dunderhead". This quirk of hers is mostly pointed out by Subaru, who has a tendency to tease her about it, though Emilia doesn't pay it any mind. Fortuna has also questioned where she had picked up such a vocabulary. Furthermore, Emilia also has a strong tendency to drag out the word "really", which can be best observed when she's talking about something dear or important to her. This quirk was likely picked up from Fortuna, who had the same verbal tic. She is rarely angered by the people around her, only having been angered by the despicable actions of Pandora, Regulus, and Aldebaran's betrayal thus far. She also has a strong belief in promises, having been raised by Fortuna to believe that if someone makes a promise, they should always keep them, something she took to heart not just with for the spirits around her but also with the people she interacts with, and it is because of this that she sometimes became emotional with Subaru, a habitual promise breaker. She is someone who believes in a person's good nature, and she displays a kindness similar to Subaru's, typically wanting to befriend others even despite race, appearance, and even position putting herself and others at odds, and she does not want to fight with others if she can help it, preferring to talk with her enemies and understand them rather than fight if it is an option. That said, if she is forced to, she will enter a fight with the intention of killing if it is necessary, though her fighting style still focuses on capturing her enemies alive, something very reminiscent of her gentle nature.</s>
                     <|user|>emilia, how do you see yourself? </s>
                     <|assistant|>""", return_tensors="pt").to(model.device)
output = model.generate(**tokenized, temperature=0.5,max_new_tokens=200)


In [ ]:
tokenizer.batch_decode(output, skip_special_tokens=True)[0]

In [ ]:
model.save_pretrained("ssl_without_ignore_index")

In [ ]:
#ignore index
model.config

In [ ]:
optimizer = AdaBelief(model.parameters(), lr=5e-5)
train_steps=100000
optimizer.zero_grad()
losses=[]
eval_losses=[]
slice_token= tokenizer("assistant", return_tensors="pt",add_special_tokens=False).input_ids[0][0]
for stp in range(train_steps):
    batch=train.sample(1)
    inputb=batch["text"].values[0]
    inputm=tokenizer(inputb, return_tensors="pt").to(model.device)
    labels = inputm.input_ids.clone()
    getpos = (inputm.input_ids == slice_token).nonzero(as_tuple=True)[1][0]
    labels[:, :getpos] = -100  # Set the position of the slice token to -100
    outputs = model(**inputm, labels=labels)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
    optimizer.step()
    optimizer.zero_grad()
    losses.append(loss.item())
    clear_output()
    with torch.no_grad():
        eval_batch= eval.sample(1)
        eval_input=eval_batch["text"].values[0]
        eval_input_ids = tokenizer(eval_input, return_tensors="pt").to(model.device)
        eval_labels = eval_input_ids.input_ids.clone()
        getpos = (eval_input_ids.input_ids == slice_token).nonzero(as_tuple=True)[1]
        eval_labels[:, getpos] = -100  # Set the position of the slice token to -100
        outputs = model(**eval_input_ids, labels=eval_labels)
        eval_loss = outputs.loss
        eval_losses.append(eval_loss.item())
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(losses, label='Training Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(eval_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Validation Loss')
    plt.legend()
    plt.show()


In [ ]:
d=tokenizer("lol <|assistant|> dou", return_tensors="pt",add_special_tokens=False).input_ids

In [ ]:
print(getpos[0])

In [ ]:
labels

In [ ]:
print(eval_labels)

In [ ]:
print(d)

In [ ]:
print(tokenizer.decode(82639))
getpos = (d == 77091).nonzero(as_tuple=True)[1]

In [ ]:
print(getpos)